In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
# Версии из отправленного архива кладутся поверх: именно на них обучена структурная модель.
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.measure_features import measures, compare_measures, MEASURE_FEATURES
from src.features import extract_model_features
from src.model import BoostedPairModel
from src.export_boost import export, save, predict_proba
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
sc = os.path.dirname(glob.glob("/kaggle/input/**/ce_relaxed.npy", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
y = (pairs["target"].to_numpy() > 0).astype(np.int8)
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
known = sorted(set(items.category.astype(str)))
log(f"пар {len(pairs):,}, доля+ {y.mean():.3f}")

names = list(feature_names(False))
X = np.zeros((len(pairs), len(names)), dtype=np.float32)
for c in known:
    rows = np.flatnonzero(cat == c)
    if not len(rows): continue
    sub = items[items.category.astype(str) == c].reset_index(drop=True)
    X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), known, with_neighbours=False)
    del sub; gc.collect()
log("парные признаки готовы")

t = time.perf_counter()
legacy = extract_model_features(pairs[["id1", "id2"]], items)
primary = BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat)
aux = BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)
structural = (0.8 * primary + 0.2 * aux).astype(np.float32)
del legacy; gc.collect()
log(f"структурная модель за {time.perf_counter()-t:.0f}с")

M = {int(i): measures(a) for i, a in zip(items.id, items.attributes)}
MX = np.array([[r[n] for n in MEASURE_FEATURES] for r in
               (compare_measures(M[x], M[z]) for x, z in zip(pairs.id1, pairs.id2))], dtype=np.float32)
CE = {n: np.load(f"{sc}/{n}.npy").astype(np.float32) for n in ("ce_relaxed", "ce_combo")}
codes = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)

BASE_COLS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "category_code"]
BASE = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], codes]).astype(np.float64)
WITH_COLS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "structural", "category_code"]
WITH = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], structural, codes]).astype(np.float64)

masks = {c: cat == c for c in np.unique(cat)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos, neg = rows[y[rows]==1], rows[y[rows]==0]
            keep = min(len(pos), max(5, int(round(rate/(1-rate)*len(neg)))))
            ch = np.concatenate([rng.choice(pos, keep, replace=False), neg])
            per.append(average_precision_score(y[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

COLS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "structural", "category_code"]
FULL = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], structural, codes]).astype(np.float64)
log(f"матрица {FULL.shape}")

# Метрика — среднее по категориям, каждая весит одинаково. В обучении крупные категории
# перевешивают: 9355 пар в Обуви против 6875 в Мебели. Выравниваем вклад.
counts = pd.Series(cat).value_counts()
balance = np.array([1.0 / counts[c] for c in cat]); balance *= len(balance) / balance.sum()

half = np.random.default_rng(5).permutation(len(y)) % 2
def honest(params, weights=None):
    p = np.zeros(len(y))
    for h in (0, 1):
        tr, te = half != h, half == h
        g = HistGradientBoostingClassifier(early_stopping=False, random_state=0, **params)
        g.fit(FULL[tr], y[tr], sample_weight=None if weights is None else weights[tr])
        p[te] = g.predict_proba(FULL[te])[:, 1]
    return macro(rk(p)), p

GRID = [
    dict(max_iter=500,  max_leaf_nodes=63,  learning_rate=0.06, l2_regularization=1.0),
    dict(max_iter=800,  max_leaf_nodes=63,  learning_rate=0.06, l2_regularization=1.0),
    dict(max_iter=1500, max_leaf_nodes=63,  learning_rate=0.04, l2_regularization=1.0),
    dict(max_iter=1500, max_leaf_nodes=255, learning_rate=0.04, l2_regularization=2.0),
    dict(max_iter=800,  max_leaf_nodes=127, learning_rate=0.05, l2_regularization=2.0),
    dict(max_iter=2000, max_leaf_nodes=31,  learning_rate=0.05, l2_regularization=1.0),
]
best = (None, -1, None)
for params in GRID:
    (mu, sd), _ = honest(params)
    log(f"  {str(params):<96} {mu:.6f} ± {sd:.6f}")
    if mu > best[1]: best = (params, mu, None)
log(f"лучшие параметры: {best[0]} -> {best[1]:.6f}")

(mu_b, sd_b), _ = honest(best[0], balance)
log(f"они же с выравниванием категорий: {mu_b:.6f} ± {sd_b:.6f}")
use_balance = mu_b > best[1]
final_params, final_score = best[0], max(best[1], mu_b)

weights = balance if use_balance else None
final = HistGradientBoostingClassifier(early_stopping=False, random_state=0, **final_params)
final.fit(FULL, y, sample_weight=weights)
trees = export(final)
save("/kaggle/working/fusion_boost.npz", trees)
ok = np.allclose(predict_proba(trees, FULL[:2000]), final.predict_proba(FULL[:2000])[:, 1], atol=1e-6)
json.dump({"columns": COLS, "categories": known, "params": final_params,
           "honest_macro": final_score, "n_train": int(len(y)),
           "uses_structural": True, "balanced_categories": bool(use_balance)},
          open("/kaggle/working/fusion_info.json", "w"), ensure_ascii=False, indent=1)
log(f"выгружено, выравнивание {use_balance}, совпадение {ok}, метрика {final_score:.6f}")
